# setup

In [1]:
import zarr
import pandas as pd
import numpy as np
import dask.array as da
import napari

VIDEOS = [
    "44b6_33b596bf", "44b6_40c45f5a", "44b6_551a5dba",
    # "44b6_87bba6c4", "44b6_abf82518", "44b6_c96cfa10",
    # "44b6_eb2880fc", "6bba_1f58c2f6", "6bba_55c70843",
    # "6bba_61dd1e0d", "6bba_6ca87370", "6bba_6feeb0b1",
    # "6bba_80d12824", "6bba_9e23430b", "6bba_debd7bfa"
]

PATCH_SIZE = (16, 64, 64) #z,y,x

# patch crop implementation

In [9]:
def get_geff(path):
    g = zarr.open(path, mode='r')
    
    points = np.column_stack([
        g["nodes/props/t/values"][:],
        g["nodes/props/z/values"][:],
        g["nodes/props/y/values"][:],
        g["nodes/props/x/values"][:],
    ])
    return points

def extract_patch(src, point, patch_size):
    t, zc, yc, xc = map(int, point)
    pz, py, px = patch_size

    # Calculate bounds
    centers = [zc, yc, xc]
    sizes = [pz, py, px]
    starts = [zc - pz // 2, yc - py // 2, xc - px // 2]
    ends = [s + n for s, n in zip(starts, sizes)]

    # Clip extraction bounds to the image boundaries
    limits = src.shape[1:]  # (Z, Y, X)
    clipped_starts = [max(0, s) for s in starts]
    clipped_ends = [min(limit, e) for limit, e in zip(limits, ends)]

    patch = src[
        t,
        clipped_starts[0]:clipped_ends[0],
        clipped_starts[1]:clipped_ends[1],
        clipped_starts[2]:clipped_ends[2],
    ]

    # Calculate padding required on each side
    pad_width = [
        (cs - s, e - ce)
        for s, e, cs, ce in zip(starts, ends, clipped_starts, clipped_ends)
    ]

    if any(before or after for before, after in pad_width):
        patch = np.pad(patch, pad_width, mode="reflect")

    center = tuple(
        c - s for c, s in zip(centers, starts)
    )

    return patch, center

In [ ]:
for i, video in enumerate(VIDEOS):
    print(f"video: {video}")
    src = zarr.open(f'../train_processed/{video}.zarr/0', mode='r')
    pts = get_geff(f'../train/{video}.geff')

    pz, py, px = PATCH_SIZE

    dst = zarr.open(f"./unet_patches/{video}.zarr", mode="w")

    patches = dst.create_array(
        "0",
        shape=(len(pts), pz, py, px),
        chunks=(1, pz, py, px),
        dtype=src.dtype,
    )

    gt_centroids = dst.create_array(
        "gt_centroids",
        shape=(len(pts), 4),
        chunks=(1, 4),
        dtype="int32",
    )

    for j, point in enumerate(pts):
        patch, center = extract_patch(src, point, PATCH_SIZE)
        patches[j] = patch
        gt_centroids[j] = (j, *center)

video: 44b6_33b596bf
video: 44b6_40c45f5a
video: 44b6_551a5dba
video: 44b6_87bba6c4
video: 44b6_abf82518
video: 44b6_c96cfa10
video: 44b6_eb2880fc
video: 6bba_1f58c2f6
video: 6bba_55c70843
video: 6bba_61dd1e0d
video: 6bba_6ca87370
video: 6bba_6feeb0b1
video: 6bba_80d12824
video: 6bba_9e23430b
video: 6bba_debd7bfa


### Napari Viz

In [2]:
for i, video in enumerate(VIDEOS):
    root = zarr.open(f"./unet_patches/{video}.zarr", mode="r")
    patches = root["0"]
    gt_centroids = np.asarray(root["gt_centroids"][:])

    viewer = napari.Viewer(ndisplay=3)

    viewer.add_image(
        patches,
        name=f"{video}_patches ",
        colormap="magma",
        scale=(1, 4, 1, 1),
        gamma=2.0,
    )
    
    viewer.add_points(
        gt_centroids,
        name="gt_centroids",
        size=5,
        face_color="cyan",
        scale=(1, 4, 1, 1),
    )

    napari.run()

could not determine DPI
could not determine DPI
